# cavity — de Vahl Davis 2D 검증 플롯
로그(`logs/cavity_<Ra>.log`)의 Nu(t) 이력과 `stats/<Ra>/` 중앙선 프로파일을 벤치마크(de Vahl Davis 1983)와 비교한다.
시각화는 show + save (`plot/*.png`).

In [ ]:
import re, glob, os
import numpy as np
import matplotlib.pyplot as plt
HERE = os.path.dirname(os.path.abspath("__file__")) if "__file__" in dir() else os.getcwd()
APP = os.path.abspath(os.path.join(HERE, ".."))
BENCH = {  # de Vahl Davis 1983: Nu_avg, u_max(z), w_max(x)
    "1e3": (1.118, 3.649, 0.813, 3.697, 0.178),
    "1e4": (2.243, 16.178, 0.823, 19.617, 0.119),
    "1e5": (4.519, 34.73, 0.855, 68.59, 0.066),
    "1e6": (8.800, 64.63, 0.850, 219.36, 0.0379),
}
CASES = ["1e3", "1e4", "1e5", "1e6"]
SUFFIX = "_mirror"   # 2차 벽(velocity_ghost = mirror) 런; "" 또는 "_value" 로 바꿔 1차 벽 런 비교

def parse_log(path):
    rows = []
    for line in open(path):
        m = re.match(r"\s*(\d+)\s+([\d.]+)\s+([\d.e+-]+)\s+([\d.e+-]+)\s+([\d.e+-]+)\s+([\d.e+-]+)\s+([\d.e+-]+)\s+\(([\d.]+)\)\s+([\d.e+-]+)\s+\(([\d.]+)\)", line)
        if m: rows.append([float(v) for v in m.groups()])
    a = np.array(rows)
    return dict(step=a[:,0], t=a[:,1], dt=a[:,2], div=a[:,3], nu_hot=a[:,4], nu_cold=a[:,5],
                u_max=a[:,6], z_u=a[:,7], w_max=a[:,8], x_w=a[:,9])

def read_sections(path):
    secs, cur = [], []
    for line in open(path):
        if line.startswith("# section"):
            if cur: secs.append(np.array(cur)); cur = []
            continue
        if line.startswith("#") or not line.strip(): continue
        cur.append([float(v) for v in line.split()])
    if cur: secs.append(np.array(cur))
    return secs


## 1. Nu(t) 이력 — 정상상태 도달 확인

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for ax, ra in zip(axes, CASES):
    logs = glob.glob(f"{APP}/logs/cavity_{ra}{SUFFIX}*.log")
    if not logs: ax.set_title(f"Ra={ra} (no log)"); continue
    L = parse_log(logs[0])
    ax.plot(L["t"], L["nu_hot"], "k-", label="Nu_hot")
    ax.plot(L["t"], L["nu_cold"], "r--", label="Nu_cold")
    ax.axhline(BENCH[ra][0], color="b", ls=":", label=f"de Vahl Davis {BENCH[ra][0]}")
    ax.set_title(f"Ra={ra}"); ax.set_xlabel("t (free-fall)"); ax.set_ylabel("Nu"); ax.legend(fontsize=7)
    ax.set_ylim(0.9*BENCH[ra][0], 1.3*BENCH[ra][0])
plt.tight_layout(); plt.savefig(f"{HERE}/nu_history.png", dpi=130); plt.show()


## 2. 최종값 vs 벤치마크 표

In [ ]:
print(f"{'Ra':>5} {'Nu':>8} {'bench':>7} {'err%':>6} | {'u_max':>8} {'bench':>7} {'err%':>6} | {'w_max':>8} {'bench':>7} {'err%':>6}")
for ra in CASES:
    logs = glob.glob(f"{APP}/logs/cavity_{ra}{SUFFIX}*.log")
    if not logs: continue
    L = parse_log(logs[0]); b = BENCH[ra]
    nu, um, wm = L["nu_hot"][-1], abs(L["u_max"][-1]), abs(L["w_max"][-1])
    print(f"{ra:>5} {nu:8.4f} {b[0]:7.3f} {100*(nu/b[0]-1):6.2f} | {um:8.3f} {b[1]:7.3f} {100*(um/b[1]-1):6.2f} | {wm:8.3f} {b[3]:7.3f} {100*(wm/b[3]-1):6.2f}   (t={L['t'][-1]:.0f})")


## 3. 중앙선 프로파일 (최종 통계 파일) — u(z) at x=½, w(x) at z=½, T

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for c, ra in enumerate(CASES):
    files = sorted(glob.glob(f"{APP}/stats/{ra}{SUFFIX}/cavity_stats_*.dat"))
    if not files: continue
    s1, s2 = read_sections(files[-1])[:2]
    b = BENCH[ra]
    ax = axes[0, c]; ax.plot(s1[:,1], s1[:,0], "k-"); ax.axvline(b[1], color="b", ls=":"); ax.axvline(-b[1], color="b", ls=":")
    ax.set_title(f"Ra={ra}: u(z) at x=1/2  (bench |u|max={b[1]})"); ax.set_xlabel("u L/alpha"); ax.set_ylabel("z")
    ax = axes[1, c]; ax.plot(s2[:,0], s2[:,1], "k-"); ax.axhline(b[3], color="b", ls=":"); ax.axhline(-b[3], color="b", ls=":")
    ax.set_title(f"w(x) at z=1/2  (bench |w|max={b[3]})"); ax.set_xlabel("x"); ax.set_ylabel("w L/alpha")
plt.tight_layout(); plt.savefig(f"{HERE}/midlines.png", dpi=130); plt.show()

fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for ax, ra in zip(axes, CASES):
    files = sorted(glob.glob(f"{APP}/stats/{ra}{SUFFIX}/cavity_stats_*.dat"))
    if not files: continue
    s1, s2 = read_sections(files[-1])[:2]
    ax.plot(s2[:,0], s2[:,2], "k-", label="T(x, z=1/2)"); ax.plot(s1[:,2], s1[:,0], "r--", label="T(x=1/2, z) vs z")
    ax.set_title(f"Ra={ra}"); ax.legend(fontsize=7); ax.set_xlabel("x  /  T"); ax.set_ylabel("T  /  z")
plt.tight_layout(); plt.savefig(f"{HERE}/temperature_midlines.png", dpi=130); plt.show()


## 4. Table V 10항목 전수 비교 (plot/benchmark.py — 최종 장 덤프에서 계산)

In [ ]:
import subprocess, sys
dumps = [f for ra in CASES for f in sorted(glob.glob(f"{APP}/stats/{ra}{SUFFIX}/cavity_field_*.dat"))[-1:]]
print(subprocess.run([sys.executable, f"{HERE}/benchmark.py", *dumps], capture_output=True, text=True).stdout)
